# Create `telemetry.agent_telemetry`

Phase 1, item 7 (`.claude/rules/telemetry.md`). Defines the **full** schema
now — every field the table will ever need, even ones that stay
null/`False`/`0`/empty until Phase 3+ populates them for real. Only the
writer code grows later; this schema is meant to need altering less than
usual, not never.

`is_projection` deliberately excluded — added *with* the Phase 7
`run_projection` tool, not before.

Companion notebook: `delete_agent_telemetry_table.ipynb` — BigQuery can't
`ALTER` a column's mode or rename it, so a real schema change means dropping
and re-running this one.

In [1]:
from google.cloud import bigquery

PROJECT_ID = "instacart-ml-model"
DATASET_ID = "telemetry"
TABLE_ID = "agent_telemetry"
LOCATION = "US"  # confirmed via `bq show` against agent_safe/vector_db/staging (2026-09-18)

client = bigquery.Client(project=PROJECT_ID)

## Schema

Three groups, matching `telemetry.md`'s own table exactly:
- **Required, real from Phase 1** — `conversation_id`, `user_id`, `question`,
  `answer_markdown`, `turn_started_at`, `turn_completed_at`.
- **Required, writer supplies a literal default (`False`/`0`) until real**
  — booleans and counters. Required because the writer always sets *some*
  value, even a placeholder — the schema doesn't encode the default itself,
  the writer code (a later step) does.
- **Nullable, genuinely absent until Phase 3** — where the field's absence
  is itself meaningful (no chart generated, no pending query).

In [2]:
TOOL_CALL_FIELDS = [
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("started_at", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("completed_at", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("args", "STRING", mode="REQUIRED"),       # json.dumps'd
    bigquery.SchemaField("query_text", "STRING", mode="NULLABLE"), # SQL/DAX only
    bigquery.SchemaField("result", "STRING", mode="REQUIRED"),     # json.dumps'd
    bigquery.SchemaField("success", "BOOLEAN", mode="REQUIRED"),
    bigquery.SchemaField("error", "STRING", mode="NULLABLE"),
]

ERROR_FIELDS = [
    bigquery.SchemaField("stage", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("error_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("message", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("occurred_at", "TIMESTAMP", mode="REQUIRED"),
]

SCHEMA = [
    # --- Required, real from Phase 1 ---
    bigquery.SchemaField("conversation_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING", mode="REQUIRED"),        # claims["oid"]
    bigquery.SchemaField("question", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("answer_markdown", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("turn_started_at", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("turn_completed_at", "TIMESTAMP", mode="REQUIRED"),

    # --- Nullable: genuinely absent until Phase 3, absence is meaningful ---
    bigquery.SchemaField("filter_context", "STRING", mode="NULLABLE"),  # json.dumps'd list[dict]
    bigquery.SchemaField("active_page", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("pending_query", "STRING", mode="NULLABLE"),   # the largest pending query
    bigquery.SchemaField("estimated_cost", "STRING", mode="NULLABLE"),  # display dollars, not numeric
    bigquery.SchemaField("approval_decision", "STRING", mode="NULLABLE"),  # "approved" | "rejected"
    bigquery.SchemaField("chart_url", "STRING", mode="NULLABLE"),

    # --- Required, writer defaults to False/0 until Phase 3 ---
    bigquery.SchemaField("verified", "BOOLEAN", mode="REQUIRED"),
    bigquery.SchemaField("verification_retry_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("length_retry_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("needs_approval", "BOOLEAN", mode="REQUIRED"),
    bigquery.SchemaField("cost_cap_exceeded", "BOOLEAN", mode="REQUIRED"),
    bigquery.SchemaField("prompt_tokens", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("completion_tokens", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("llm_calls", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("bytes_consumed", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("iteration_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("cancelled", "BOOLEAN", mode="REQUIRED"),
    bigquery.SchemaField("iteration_cap_hit", "BOOLEAN", mode="REQUIRED"),

    # --- Repeated: empty by default, no NULLABLE/REQUIRED distinction applies ---
    bigquery.SchemaField("tool_calls", "RECORD", mode="REPEATED", fields=TOOL_CALL_FIELDS),
    bigquery.SchemaField("errors", "RECORD", mode="REPEATED", fields=ERROR_FIELDS),
    bigquery.SchemaField("suggested_follow_ups", "STRING", mode="REPEATED"),
    bigquery.SchemaField("pending_queries", "STRING", mode="REPEATED"),  # full list, BigQuery
    bigquery.SchemaField("deferred_dax", "STRING", mode="REPEATED"),     # full list, DAX
]

print(f"{len(SCHEMA)} top-level fields, {len(TOOL_CALL_FIELDS)} nested in tool_calls, "
      f"{len(ERROR_FIELDS)} nested in errors")

29 top-level fields, 8 nested in tool_calls, 4 nested in errors


## Create the dataset, then the table

`exists_ok=True` on both — safe to re-run. Note this does **not** alter an
already-existing table's schema if one's already there with different
fields; that's a deliberate follow-up step (`ALTER TABLE` / a migration
notebook), not something to silently paper over here.

In [3]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset ready: {dataset.dataset_id} ({dataset.location})")

Dataset ready: telemetry (US)


In [4]:
table_ref = bigquery.Table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}", schema=SCHEMA)
table = client.create_table(table_ref, exists_ok=True)
print(f"Table ready: {table.project}.{table.dataset_id}.{table.table_id}")
print(f"{len(table.schema)} top-level fields, {table.num_rows} rows")

Table ready: instacart-ml-model.telemetry.agent_telemetry
29 top-level fields, 0 rows


## Verify — print the real schema back from BigQuery, not just what we sent

In [6]:
fetched = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
for field in fetched.schema:
    if field.field_type == "RECORD":
        print(f"{field.name} ({field.mode} {field.field_type}):")
        for sub in field.fields:
            print(f"    {sub.name} ({sub.mode} {sub.field_type})")
    else:
        print(f"{field.name} ({field.mode} {field.field_type})")

conversation_id (REQUIRED STRING)
user_id (REQUIRED STRING)
question (REQUIRED STRING)
answer_markdown (REQUIRED STRING)
turn_started_at (REQUIRED TIMESTAMP)
turn_completed_at (REQUIRED TIMESTAMP)
filter_context (NULLABLE STRING)
active_page (NULLABLE STRING)
pending_query (NULLABLE STRING)
estimated_cost (NULLABLE STRING)
approval_decision (NULLABLE STRING)
chart_url (NULLABLE STRING)
verified (REQUIRED BOOLEAN)
verification_retry_count (REQUIRED INTEGER)
length_retry_count (REQUIRED INTEGER)
needs_approval (REQUIRED BOOLEAN)
cost_cap_exceeded (REQUIRED BOOLEAN)
prompt_tokens (REQUIRED INTEGER)
completion_tokens (REQUIRED INTEGER)
llm_calls (REQUIRED INTEGER)
bytes_consumed (REQUIRED INTEGER)
iteration_count (REQUIRED INTEGER)
cancelled (REQUIRED BOOLEAN)
iteration_cap_hit (REQUIRED BOOLEAN)
tool_calls (REPEATED RECORD):
    name (REQUIRED STRING)
    started_at (REQUIRED TIMESTAMP)
    completed_at (REQUIRED TIMESTAMP)
    args (REQUIRED STRING)
    query_text (NULLABLE STRING)
    r

## Test the writer — dummy row, awaited insert, timed

`asyncio.to_thread` hands the blocking `google-cloud-bigquery` call to a
worker thread, but `await` still blocks the caller until it finishes — this
is *not* backgrounding. Cloud Run sees the request as in-flight the whole
time; the response can't be sent until this returns either way.

In [ ]:
import asyncio
import time
import uuid
from datetime import datetime, timezone


def build_telemetry_row(
    conversation_id: str,
    user_id: str,
    question: str,
    answer_markdown: str,
    turn_started_at: datetime,
    turn_completed_at: datetime,
) -> dict:
    """Full agent_telemetry row — the six Phase 1 fields real, everything
    else the documented placeholder default until the phase that populates
    it lands (telemetry.md)."""
    return {
        "conversation_id": conversation_id,
        "user_id": user_id,
        "question": question,
        "answer_markdown": answer_markdown,
        "turn_started_at": turn_started_at.isoformat(),
        "turn_completed_at": turn_completed_at.isoformat(),
        "filter_context": None,
        "active_page": None,
        "pending_query": None,
        "pending_queries": [],
        "deferred_dax": [],
        "estimated_cost": None,
        "approval_decision": None,
        "chart_url": None,
        "verified": False,
        "verification_retry_count": 0,
        "length_retry_count": 0,
        "needs_approval": False,
        "cost_cap_exceeded": False,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "llm_calls": 0,
        "bytes_consumed": 0,
        "iteration_count": 0,
        "cancelled": False,
        "iteration_cap_hit": False,
        "tool_calls": [],
        "errors": [],
        "suggested_follow_ups": [],
    }


async def insert_telemetry_row(row: dict) -> list:
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
    return await asyncio.to_thread(client.insert_rows_json, table_ref, [row])


test_row = build_telemetry_row(
    conversation_id=str(uuid.uuid4()),
    user_id="test-user-oid",
    question="dummy test question",
    answer_markdown="dummy test answer",
    turn_started_at=datetime.now(timezone.utc),
    turn_completed_at=datetime.now(timezone.utc),
)

start = time.perf_counter()
errors = await insert_telemetry_row(test_row)
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Insert took {elapsed_ms:.1f}ms")
print("ERRORS:" if errors else "No errors.", errors if errors else "")
print("conversation_id:", test_row["conversation_id"])

## Verify the row landed, then clean up

`SELECT` visibility on a streamed row is near-immediate — this is checking
the row actually arrived, separate from whether `insert_rows_json` reported
any errors above.

In [12]:
select_query = f"""
SELECT conversation_id, user_id, question, answer_markdown, verified, tool_calls
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE conversation_id = @conversation_id
"""
job = client.query(
    select_query,
    job_config=bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("conversation_id", "STRING", test_row["conversation_id"])
    ]),
)
rows = list(job.result())
print(f"{len(rows)} row(s) found")
for r in rows:
    print(dict(r))

1 row(s) found
{'conversation_id': '6de7f0b3-7b41-4e21-9e58-4a758a599a00', 'user_id': 'test-user-oid', 'question': 'dummy test question', 'answer_markdown': 'dummy test answer', 'verified': False, 'tool_calls': []}


In [13]:
try:
    delete_query = f"""
    DELETE FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE conversation_id = @conversation_id
    """
    job = client.query(
        delete_query,
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("conversation_id", "STRING", test_row["conversation_id"])
        ]),
    )
    job.result()
    print("Deleted.")
except Exception as e:
    print("Delete failed — expected if the row is still in the streaming buffer, not a bug:")
    print(e)
    print("\nFallback: drop and recreate the table instead (safe, it's empty otherwise).")
    print("client.delete_table(f\"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}\")")
    print("Then re-run the create-dataset/create-table cells above.")

Delete failed — expected if the row is still in the streaming buffer, not a bug:
400 GET https://bigquery.googleapis.com/bigquery/v2/projects/instacart-ml-model/queries/71eee474-22ea-42da-a205-b9cdf52203a6?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table instacart-ml-model.telemetry.agent_telemetry would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 71eee474-22ea-42da-a205-b9cdf52203a6


Fallback: drop and recreate the table instead (safe, it's empty otherwise).
client.delete_table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
Then re-run the create-dataset/create-table cells above.
